# Banc d'essai `bench` — génération AP-HP

Notebook du nouveau monde `work_prompts/` (spec : `docs/spec_testrun_run_stage.md`, v3.6),
organisé selon le **cycle de vie d'un test** : initialisation → préparation →
run → bilan. Toutes les étapes qui touchent le disque sont **idempotentes** :
elles s'exécutent si nécessaire, sinon elles skippent avec un message d'état —
le notebook se ré-exécute intégralement sur un test déjà préparé.

Deux gestes à ne pas confondre :

- **MONTAGE** (section 2) : installer le **jeu de templates** du test dans
  `system/one_gen/` depuis la version précédente — c'est l'objet versionné,
  la chaîne des tests est la chaîne des versions des jeux.
- **SEEDING puis FIGEMENT** (section 3.1) : créer les **dossiers CRH** depuis
  les scénarios, puis résoudre le jeu par famille (`template.txt`) en
  `prompt_system_one_gen.txt` dans chaque dossier — c'est la production du run.

Le jeu peut porter un `prefix.txt` : au seeding, il remplace alors le
prefix fictomed des scénarios — le prefill suit la chaîne des versions,
comme le reste du jeu.

Rien ne se nettoie : pour repartir de zéro, on crée `tests/NN+1`. Le disque
fait foi ; **tout `work_prompts/` est versionné et partagé** (§9) — « figer »
un jeu = commiter son test.

La clé API vient **exclusivement** de l'environnement (`MISTRAL_API_KEY`),
jamais du notebook. Tout s'exécute **sans clé** jusqu'aux dry-runs inclus ;
seules les cellules « run réel » l'exigent.


## 1. Initialisation — déclarations pures

Bootstrap (`sys.path`, import de `bench`), paramétrage du test courant,
paramètres de sélection et de génération, client Mistral à la demande.
Aucune exécution lourde ici.


In [77]:
from __future__ import annotations

from pathlib import Path
import json
import os
import shutil
import sys

import polars as pl
from IPython.display import display


def _find_repo_root(start: Path) -> Path:
    """Racine du repo Stream : le dossier qui contient `bench/` et `core/`."""
    for candidate in (start, *start.parents):
        if (candidate / "bench").is_dir() and (candidate / "core").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Racine du repo Stream introuvable depuis {start} — lancer le "
        "notebook depuis work_prompts/ (ou un sous-dossier du repo)."
    )


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import bench
from bench import (
    BenchError,
    Pricing,
    copy_system_prompts,
    generate,
    load_reports,
    scenario_dirs,
    seed_user_prompts,
    summarize_costs,
    user_from_column,
    write_prompts,
)



def show_first_prompt(result, *, max_chars: int = 6000) -> None:
    """Affiche le premier prompt assemblé d'un `GenResult` dry-run.

    Contrôle à sec ET test de complétude : si `generate` a rendu la main,
    aucun fichier ne manquait dans aucun dossier scénario (§3.5).
    """
    if result.reports.height == 0:
        print("Aucun scénario retenu.")
        return
    row = result.reports.row(0, named=True)
    print(f"Scénario : {row['scenario']} — famille : {row['template']}")
    for label, key in (
        ("PROMPT SYSTÈME", "system_prompt"),
        ("PROMPT USER", "user_prompt"),
        ("PREFIX (assistant)", "prefix"),
    ):
        text = row[key] or ""
        print(f"\n{'=' * 28} {label} — {len(text)} caractère(s) {'=' * 28}")
        print(text[:max_chars])
        if len(text) > max_chars:
            print(f"[... tronqué à {max_chars} caractères]")


print("Racine repo  :", REPO_ROOT)
print("bench        :", Path(bench.__file__).resolve().parent)


Racine repo  : /Users/remi/Documents/Stream
bench        : /Users/remi/Documents/Stream/bench


### Paramètres de génération


In [78]:
MODEL = os.environ.get("MISTRAL_MODEL", "mistral-large-latest")
MAX_TOKENS_SUMMARY = 8_000
MAX_TOKENS_CR = 128_000
# Transport des appels Mistral (spec v3.7) : "sync" = un chat.complete par
# scénario, MAX_WORKERS en parallèle ; "batch" = 50 % moins cher mais reste en
# file indéfiniment depuis septembre 2026 — à réactiver quand Mistral l'aura
# rétabli. Les tarifs (USD / 1M tokens, Mistral Large, https://mistral.ai/pricing)
# suivent le transport.
TRANSPORT = os.environ.get("MISTRAL_TRANSPORT", "sync")
MAX_WORKERS = 3
_TARIFS = {"sync": (0.5, 1.5), "batch": (0.25, 0.75)}
PRICING = Pricing(*_TARIFS[TRANSPORT])

# Prefill de la première génération d'un test deux temps (annexe 2-gen, non
# utilisé — repris de l'ancien notebook).
FIRST_GEN_PREFIX = "Résumé clinique :"

# Injection du résumé intermédiaire dans le user prompt du second temps
# (annexe 2-gen, §4).
SUMMARY_HEADER = """### RÉSUMÉ CLINIQUE ISSU DE LA PREMIÈRE GÉNÉRATION
Le résumé ci-dessous est une aide intermédiaire.
Le scénario clinique, les codes, les fiches descriptives et les instructions
restent prioritaires en cas de divergence."""

SUMMARY_FOOTER = "### FIN DU RÉSUMÉ CLINIQUE INTERMÉDIAIRE"

# Test 3 — vérificateur : textes d'exemple, à adapter à la campagne.
VERIF_SYSTEM = """Tu es un médecin DIM. On te fournit un compte rendu
hospitalier généré automatiquement. Vérifie sa cohérence clinique et sa
conformité aux règles de codage, puis rends un verdict structuré :
CONFORME ou NON CONFORME, suivi de la liste des anomalies constatées."""

VERIF_USER = "Vérifie le compte rendu suivant et rends ton verdict."
VERIF_HEADER = "### COMPTE RENDU À VÉRIFIER"
VERIF_FOOTER = "### FIN DU COMPTE RENDU"

print("Modèle :", MODEL)
print("Transport :", TRANSPORT, "— tarifs ($ / 1M tokens) :", PRICING)

Modèle : mistral-large-latest
Tarifs batch ($ / 1M tokens) : Pricing(batch_input_usd_per_million=0.25, batch_output_usd_per_million=0.75)


### Clé API — jamais en dur

La clé vient **exclusivement** de `os.environ["MISTRAL_API_KEY"]`, exportée **avant**
le lancement du kernel (`export MISTRAL_API_KEY=...` puis relancer Jupyter). Elle ne
doit jamais apparaître dans le notebook, ni dans aucun fichier versionné.

Le client n'est construit qu'au moment d'un **run réel** : les cellules dry-run
n'ont pas besoin de clé.

In [79]:
def mistral_client():
    """Client Mistral construit à la demande — uniquement pour les runs réels."""
    try:
        api_key = os.environ["MISTRAL_API_KEY"]
    except KeyError:
        raise RuntimeError(
            "Variable d'environnement MISTRAL_API_KEY absente.\n"
            "Exporter la clé AVANT de lancer le kernel :\n"
            "    export MISTRAL_API_KEY=...    # puis relancer jupyter\n"
            "La clé ne doit JAMAIS être écrite dans ce notebook ni dans un "
            "fichier versionné."
        ) from None
    if not api_key.strip():
        raise RuntimeError("MISTRAL_API_KEY est définie mais vide.")
    from core.clients import MistralClient

    return MistralClient(api_key=api_key)


print("MISTRAL_API_KEY présente :", "MISTRAL_API_KEY" in os.environ)

MISTRAL_API_KEY présente : True


## 2. Préparation des données


### 2.1 Import des données

Les parquets de `data/aphp` contiennent des **profils sources** (PMSI), pas des
scénarios : c'est la chaîne fictomed qui produit les scénarios
(`generation_id`, `template_name`, `user_prompt`, `prefix`, ...). Elle est
entièrement **locale** — Mistral n'intervient qu'aux `generate()`, aucune clé
n'est nécessaire ici. Les fonctions sont importées de
`work_modif_prompts/aphp_generation_utils.py`, réutilisées **telles quelles** ;
leurs fichiers de travail vont sous `TD/.fictomed/` (dossier caché, hors
découverte). La frontière est nette : fictomed produit les scénarios, `bench`
commence après.

> **Prérequis** : fictomed installé en **éditable** depuis le clone
> `work_modif_prompts/_dependencies/fictomed_prompt_work` (branche
> `prompt-work`), comme le faisait le §1 de l'ancien notebook :
> `pip install -e work_modif_prompts/_dependencies/fictomed_prompt_work`.
> Le paquet PyPI `fictomed` 0.1.2 n'embarque pas `regles_atih.yml` et fait
> échouer la génération ; après un `uv sync` (qui réinstalle la version
> PyPI), refaire l'installation éditable.

fictomed lit ses chemins (profils d'entrée, sorties, référentiels) dans un
fichier de config `servers.yaml`, **généré ici pour le run** sous
`TD/.fictomed/`. Pendant la génération, le fichier de profils actif est
temporairement remplacé par les candidats sélectionnés puis **restauré**
(bloc `finally` déjà dans `generate_and_select_fictomed_scenarios`, backup
sous `.fictomed/_backups/`).

In [80]:
# Paramètres de la sélection — à renseigner, puis à garder IDENTIQUES d'un
# test à l'autre de la chaîne (comparabilité).
SOURCE_PROFILES_PATH = REPO_ROOT / "data/aphp/scenarios_bn_all_20260128.pq"  # à renseigner

# Import des utilitaires de l'ancien monde et du parquet source (une fois).
if "aphp_utils" not in globals():
    import importlib.util

    _spec_utils = importlib.util.spec_from_file_location(
        "aphp_generation_utils",
        REPO_ROOT / "work_modif_prompts" / "aphp_generation_utils.py",
    )
    aphp_utils = importlib.util.module_from_spec(_spec_utils)
    _prev_dwb = sys.dont_write_bytecode
    sys.dont_write_bytecode = True  # pas de __pycache__ dans l'ancien monde
    try:
        _spec_utils.loader.exec_module(aphp_utils)
    finally:
        sys.dont_write_bytecode = _prev_dwb
    print("Utilitaires amont importés depuis :", _spec_utils.origin)

if "source_df" not in globals():
    source_df = pl.read_parquet(aphp_utils.resolve_parquet_path(SOURCE_PROFILES_PATH))
    print("Source :", SOURCE_PROFILES_PATH)
    print("Shape  :", source_df.shape, "— colonnes :", source_df.columns)




# identification des principales filières (à reporter au niveau scénario)
* Médecine 
    - Séances simples (radiothérapie, dialyse, aphérèse) :  CMD 28 à l'exclusion des autres types de séances
    - Séance polysomno : CMD 28, DP = Z04801
    - Séance Chimiothérapie simple adulte :Age >= 18, CMD 28, DP = Z511
    - HDJ Médecine adultes : Type hospitalisation = HDJ, Type GHM M ou Z
    - Médecine adultes > 3 Nuits: Durée  3, Type GHM M ou Z
    - Médecine adultes < 3 Nuits: Durée < 3, Type GHM M ou 
* Chirurige et interventionnel
    - Interventionnel adultes < 3 Nuits: Durée < 3, Type GHM K
    - Chirurgie adultes < 3 Nuits : Durée < 3, Type GHM C
    - Inteventionnel adultes > 3 Nuits: Durée >3 jours, Type GHM K
    - Chirurgie adultes > 3 Nuits: Durée >3 jours, Type GHM C
* Séjours complexes
    - Greffes de moelle, Car-T Cells adultes: 27Z02, 27Z03
    - Transplantations: racine 27C02, 27C03, 27C04, 27C05, 27C06, 27C07
    - Brulés
 * Obstétrique 
    - IVG :  14Z08Z
    - IMG & Fausse couches : Racine 14Z04,14C05, 14C06, 14C09, 14Z10, 14Z15,14Z09  
    - Accouchement normal mère : GHM 14C07A,14C08A, 14Z10, 14Z11A,14Z12A,14Z13A,14Z13T,14Z14A,14Z14T  
    - Accouchement pathologique mère: racine 14C03, racine GHM 14C03,14C07,14C08, 14Z10, 14Z11,14Z12,14Z13,14Z14 + sévérité non A et non T 
* Néonatalogie
    - Bébé normal :GHM 15M05A, 15M06A, 15M07A, 15M08A, 15M09A, 15M10A ,15M11A, 15M13A, 15M14A
    - Bébé néonat med: racine 15M05, 15M06, 15M07, 15M08, 15M09, 15M10 ,15M11, 15M13, 15M14
    - Bébé néonat chir: racine 15C02, 15C03, 15C04, 15C05, 15C06, 15M10 ,15M11, 15M13, 15M14
    - Autre néonat : racine 15M02,15M03, 15M04

Création de 2 variables
- TPEC — Type de prise en charge. La variable agrégée de votre typologie des séjours, à 6 modalités : Médecine, Chirurgie et interventionnel, Séjours complexes, Obstétrique, Néonatalogie, Autre. C'est le niveau « filière ».
- DPEC — Détail de la prise en charge. La variable fine de la même typologie (une vingtaine de modalités) : « Médecine adultes > 3 nuits », « HDJ médecine adultes », « Chirurgie adultes < 3 nuits », « Accouchement normal mère », « Bébé normal », « IVG », « Séances simples »... Chaque DPEC appartient à un TPEC (c'est le mapping DPEC_TO_TPEC de la cellule typologie). Les deux sont calculées par with_typologie(df) à partir du GHM (CMD, type, sévérité), de la racine, du DP, de la durée, du mode d'hospitalisation et de l'âge.

In [81]:
# Typologie des séjours — TPEC (type de prise en charge) / DPEC (détail).
# CMD = 2 premiers caractères du GHM ; type GHM = 3e ; sévérité = dernier.
# L'ordre des .when() fait la précédence : le spécifique (séances, obstétrique,
# néonat, séjours complexes) AVANT le tout-venant médecine/chirurgie, sinon les
# GHM 14Z/15M/27Z/28Z seraient avalés par « type M ou Z ».

RACINES_GREFFES_CART = ["27Z02", "27Z03"]
RACINES_TRANSPLANT = ["27C02", "27C03", "27C04", "27C05", "27C06", "27C07"]
RACINES_IMG_FC = ["14Z04", "14C05", "14C06", "14C09", "14Z10", "14Z15", "14Z09"]
GHM_ACC_NORMAL = ["14C03A","14C07A", "14C08A", "14Z11A", "14Z12A",
                  "14Z13A", "14Z13T", "14Z14A", "14Z14T"]
RACINES_ACC_PATHO = ["14C07", "14C08", "14Z10", "14Z11", "14Z12", "14Z13", "14Z14"]
GHM_BB_NORMAL = ["15M05A", "15M06A", "15M07A", "15M08A", "15M09A",
                 "15M10A", "15M11A", "15M13A", "15M14A"]
RACINES_BB_MED = ["15M05", "15M06", "15M07", "15M08", "15M09",
                  "15M10", "15M11", "15M13", "15M14"]
RACINES_BB_CHIR = ["15C02", "15C03", "15C04", "15C05", "15C06",
                   "15M10", "15M11", "15M13", "15M14"]
RACINES_AUTRE_NEONAT = ["15M02", "15M03", "15M04"]

DPEC_TO_TPEC = {
    "Séances simples": "Médecine",
    "Séance polysomno": "Médecine",
    "Séance chimiothérapie simple adulte": "Médecine",
    "HDJ médecine adultes": "Médecine",
    "Médecine adultes > 3 nuits": "Médecine",
    "Médecine adultes < 3 nuits": "Médecine",
    "Interventionnel adultes < 3 nuits": "Chirurgie et interventionnel",
    "Chirurgie adultes < 3 nuits": "Chirurgie et interventionnel",
    "Interventionnel adultes > 3 nuits": "Chirurgie et interventionnel",
    "Chirurgie adultes > 3 nuits": "Chirurgie et interventionnel",
    "Greffes de moelle, CAR-T Cells": "Séjours complexes",
    "Transplantations": "Séjours complexes",
    "Brûlés": "Séjours complexes",
    "IVG": "Obstétrique",
    "IMG & fausses couches": "Obstétrique",
    "Accouchement normal mère": "Obstétrique",
    "Accouchement pathologique mère": "Obstétrique",
    "Bébé normal": "Néonatalogie",
    "Bébé néonat med": "Néonatalogie",
    "Bébé néonat chir": "Néonatalogie",
    "Autre néonat": "Néonatalogie",
    "Autre": "Autre",
}


def with_typologie(df: pl.DataFrame) -> pl.DataFrame:
    """Ajoute TPEC/DPEC — colonnes requises : ghm2, racine, diag2, duree,
    mode_hospit, agean. Applicable au parquet source ; à reporter au niveau
    scénario (mêmes noms de colonnes dans le DataFrame fictomed)."""
    cmd = pl.col("ghm2").str.slice(0, 2)
    type_ghm = pl.col("ghm2").str.slice(2, 1)
    sev = pl.col("ghm2").str.slice(-1)
    racine = pl.col("racine")
    dp = pl.col("diag2")
    duree = pl.col("duree")
    adulte = pl.col("agean") >= 18
    hdj = pl.col("mode_hospit") == "HP"

    dpec = (
        # --- Séjours complexes (CMD 27, 22)
        pl.when(racine.is_in(RACINES_GREFFES_CART))
        .then(pl.lit("Greffes de moelle, CAR-T Cells"))
        .when(racine.is_in(RACINES_TRANSPLANT)).then(pl.lit("Transplantations"))
        .when(cmd == "22").then(pl.lit("Brûlés"))  # critère à confirmer (CMD 22)
        # --- Obstétrique
        .when(pl.col("ghm2") == "14Z08Z").then(pl.lit("IVG"))
        .when(racine.is_in(RACINES_IMG_FC)).then(pl.lit("IMG & fausses couches"))
        .when(pl.col("ghm2").is_in(GHM_ACC_NORMAL))
        .then(pl.lit("Accouchement normal mère"))
        .when(((racine.is_in(RACINES_ACC_PATHO) & ~sev.is_in(["A", "T"]))))
        .then(pl.lit("Accouchement pathologique mère"))
        # --- Néonatalogie
        .when(pl.col("ghm2").is_in(GHM_BB_NORMAL)).then(pl.lit("Bébé normal"))
        .when(racine.is_in(RACINES_BB_MED)).then(pl.lit("Bébé néonat med"))
        .when(racine.is_in(RACINES_BB_CHIR)).then(pl.lit("Bébé néonat chir"))
        .when(racine.is_in(RACINES_AUTRE_NEONAT)).then(pl.lit("Autre néonat"))
        # --- Médecine : séances (les spécifiques avant le tout-venant CMD 28)
        .when((cmd == "28") & (dp == "Z04801")).then(pl.lit("Séance polysomno"))
        .when((cmd == "28") & (dp == "Z511") & adulte)
        .then(pl.lit("Séance chimiothérapie simple adulte"))
        .when(cmd == "28").then(pl.lit("Séances simples"))
        # --- Médecine hors séances (HDJ d'abord, puis durée ; borne : 3 nuits
        #     et plus => « > 3 nuits », pour ne pas laisser duree == 3 sans case)
        .when(hdj & type_ghm.is_in(["M", "Z"])).then(pl.lit("HDJ médecine adultes"))
        .when((duree >= 3) & type_ghm.is_in(["M", "Z"]))
        .then(pl.lit("Médecine adultes > 3 nuits"))
        .when((duree < 3) & type_ghm.is_in(["M", "Z"]))
        .then(pl.lit("Médecine adultes < 3 nuits"))
        # --- Chirurgie et interventionnel (même borne à 3)
        .when((duree < 3) & (type_ghm == "K"))
        .then(pl.lit("Interventionnel adultes < 3 nuits"))
        .when((duree < 3) & (type_ghm == "C")).then(pl.lit("Chirurgie adultes < 3 nuits"))
        .when((duree >= 3) & (type_ghm == "K"))
        .then(pl.lit("Interventionnel adultes > 3 nuits"))
        .when((duree >= 3) & (type_ghm == "C")).then(pl.lit("Chirurgie adultes > 3 nuits"))
        .otherwise(pl.lit("Autre"))
    )
    return df.with_columns(dpec.alias("DPEC")).with_columns(
        pl.col("DPEC").replace_strict(DPEC_TO_TPEC, default="Autre").alias("TPEC")
    )


source_df = with_typologie(source_df)
display(
    source_df.group_by("TPEC", "DPEC").len()
    .sort(["TPEC", "DPEC"])
)


TPEC,DPEC,len
str,str,u32
"""Autre""","""Autre""",22
"""Chirurgie et interventionnel""","""Chirurgie adultes < 3 nuits""",618
"""Chirurgie et interventionnel""","""Chirurgie adultes > 3 nuits""",278
"""Chirurgie et interventionnel""","""Interventionnel adultes < 3 nu…",390
"""Chirurgie et interventionnel""","""Interventionnel adultes > 3 nu…",66
…,…,…
"""Néonatalogie""","""Bébé normal""",122
"""Néonatalogie""","""Bébé néonat med""",256
"""Obstétrique""","""Accouchement normal mère""",925


### 2.2 Fonction de sélection des scénarios

À garder **identiques d'un test à l'autre** : la comparabilité de la chaîne
repose sur la même graine (`SOURCE_FILTERS`, `RANDOM_SEED`, `TARGET_N`, ...).

Sélection des séjours :
- QUOTAS. Le dictionnaire {modalité: effectif} que vous renseignez au paramétrage du run : combien de séjours vous voulez dans le tirage final, strate par strate — c'est l'entrée de tirage_stratifie. QUOTAS_BY dit si les clés sont des modalités de DPEC (défaut) ou de TPEC. Votre run actuel : 15 séjours répartis sur 10 strates (2 Séances simples, 2 HDJ médecine, ... 1 IVG). Un quota à 0 documente une strate volontairement exclue ; la somme des quotas = la taille du test.


In [82]:
# Tirage stratifié — effectifs finaux par catégorie de la typologie.
# Se substitue aux étapes filtres + échantillon de prepare_source_candidates :
# le pool candidat EST le tirage final (un scénario fictomed par ligne).



def _ensure_source_ids(df: pl.DataFrame) -> pl.DataFrame:
    """Réplique l'identification de prepare_source_candidates (traçabilité aval)."""
    if "source_row_id" not in df.columns:
        df = df.with_row_index("source_row_id")
    if "source_scenario_id" not in df.columns:
        df = df.with_columns(
            (
                pl.lit(SOURCE_PROFILES_PATH.stem + "_row_")
                + pl.col("source_row_id").cast(pl.Utf8).str.zfill(7)
            ).alias("source_scenario_id")
        )
    return df


def tirage_stratifie(
    df: pl.DataFrame,
    quotas: dict[str, int],
    *,
    by: str = "DPEC",
    seed: int = 42,
) -> pl.DataFrame:
    """Tire au sort `quotas[modalité]` lignes dans chaque strate de `by`.

    Refuse si une modalité est inconnue ou si l'effectif disponible est
    insuffisant. Retourne la concaténation des strates (ordre des quotas).
    """
    dispo = dict(df.group_by(by).len().iter_rows())
    absentes_zero = [m for m, n in quotas.items() if n == 0 and m not in dispo]
    if absentes_zero:
        print(f"(quotas à 0 sur modalités absentes de {by} — sans effet : "
              f"{absentes_zero})")
    manquants = {m: (n, dispo.get(m, 0)) for m, n in quotas.items()
                 if n > 0 and dispo.get(m, 0) < n}
    if manquants:
        raise ValueError(
            f"Effectifs insuffisants (demandé, disponible) : {manquants} — "
            f"modalités disponibles : {sorted(dispo)}"
        )
    parts = [
        df.filter(pl.col(by) == m).sample(n=n, with_replacement=False,
                                          shuffle=True, seed=seed)
        for m, n in quotas.items() if n > 0
    ]
    tirage = pl.concat(parts)
    print(f"Tirage stratifié par {by} : {tirage.height} séjours "
          f"({len(parts)} strate(s), seed={seed}).")
    return tirage


In [83]:
SOURCE_FILTERS: list[dict] = [
   #{"column": "racine", "in": "endswith", "value": "8"},
]
SCENARIO_FILTERS: list[dict] = [
    # ex. {"column": "template_name", "op": "eq", "value": "surgery_outpatient.txt"},
]
RANDOM_SELECTION = True
RANDOM_SEED = 42

# Enrichissement des scénarios (lot E1 — work_prompts/enrichissement) :
# codes DAS tabac/alcool/corpulence + contexte patient (taille, poids, IMC,
# statuts), appliqué au pool candidat AVANT fictomed.
from work_prompts.enrichissement import Politique
from work_prompts.enrichissement.integration_stream import (
    enrichir_candidats,
    user_fn_enrichi,
)

ENRICHIR_SCENARIOS = True
ENRICHISSEMENT_SEED = RANDOM_SEED
POLITIQUE_ENRICHISSEMENT = Politique()


# Colonnes du tableau de contrôle du tirage (récap avant écriture) —
# retenues après inspection du schéma fictomed (55 colonnes) :
# identifiant, famille, mode de prise en charge (code), âge, sexe (1/2,
# codage PMSI), diagnostic principal (libellé + code), nb de DAS.
RECAP_TIRAGE_COLUMNS = [
    "generation_id",
    "template_name",
    "case_management_type",  # code du mode de prise en charge (libellé souvent vide)
    "age",
    "sexe",
    "icd_primary_description",
    "icd_primary_code",
    "nb_associated",
]

### 2.3 Constitution du pool candidat — filtre, tirage stratifié, enrichissement

Le pool candidat sort de cette section **prêt pour fictomed** : séjours
filtrés, tirés par quotas (`QUOTAS`, effectifs par strate TPEC/DPEC), puis
**enrichis** (codes DAS tabac/alcool/corpulence + contexte patient — lot E1).
La génération (3.1a) ne fait que le consommer : elle refuse un pool non
enrichi quand `ENRICHIR_SCENARIOS` est actif.


In [84]:
# Filtre du run : ne garder que les séjours dont le DP se termine par « 8 »
# (sous-catégories « autres formes précisées »). Les identifiants de
# traçabilité sont posés AVANT le filtre — source_row_id doit rester celui
# du parquet complet.
source_df = _ensure_source_ids(source_df)
_avant = source_df.height
source_df = source_df.filter(pl.col("diag2").str.ends_with("8"))
print(f"Filtre DP terminant par 8 : {_avant} -> {source_df.height} séjours")
display(source_df.group_by("TPEC", "DPEC").len().sort(["TPEC", "DPEC"]))

Filtre DP terminant par 8 : 7716 -> 7716 séjours


TPEC,DPEC,len
str,str,u32
"""Autre""","""Autre""",22
"""Chirurgie et interventionnel""","""Chirurgie adultes < 3 nuits""",618
"""Chirurgie et interventionnel""","""Chirurgie adultes > 3 nuits""",278
"""Chirurgie et interventionnel""","""Interventionnel adultes < 3 nu…",390
"""Chirurgie et interventionnel""","""Interventionnel adultes > 3 nu…",66
…,…,…
"""Néonatalogie""","""Bébé normal""",122
"""Néonatalogie""","""Bébé néonat med""",256
"""Obstétrique""","""Accouchement normal mère""",925


In [85]:
QUOTAS_BY = "DPEC"  # ou "TPEC"
QUOTAS: dict[str, int] = {
    # ex. par DPEC — à renseigner :
     "Séances simples": 2,
     "HDJ médecine adultes": 2,
     "Médecine adultes > 3 nuits": 2,
     "Chirurgie adultes < 3 nuits": 2,
     "Chirurgie adultes > 3 nuits": 2,
     "Interventionnel adultes < 3 nuits":1,
     "Interventionnel adultes > 3 nuits":1,
     "Accouchement normal mère": 1,
     "Bébé normal": 0,
     "IMG & fausses couches":1,
     "IVG": 0,  # DP en 8 : aucune IVG disponible
     "Greffes de moelle, CAR-T Cells": 0,
     "Brûlés" : 0,
     "Transplantations" : 0,
     
}

if QUOTAS:
    candidate_source = tirage_stratifie(
        _ensure_source_ids(source_df), QUOTAS, by=QUOTAS_BY, seed=RANDOM_SEED
    )
    display(candidate_source.group_by("TPEC", "DPEC").len().sort(["TPEC", "DPEC"]))
else:
    print("QUOTAS vide — à renseigner : la cellule de tirage en a besoin.")

(quotas à 0 sur modalités absentes de DPEC — sans effet : ['IVG', 'Greffes de moelle, CAR-T Cells', 'Brûlés', 'Transplantations'])
Tirage stratifié par DPEC : 14 séjours (9 strate(s), seed=42).


TPEC,DPEC,len
str,str,u32
"""Chirurgie et interventionnel""","""Chirurgie adultes < 3 nuits""",2
"""Chirurgie et interventionnel""","""Chirurgie adultes > 3 nuits""",2
"""Chirurgie et interventionnel""","""Interventionnel adultes < 3 nu…",1
"""Chirurgie et interventionnel""","""Interventionnel adultes > 3 nu…",1
"""Médecine""","""HDJ médecine adultes""",2
"""Médecine""","""Médecine adultes > 3 nuits""",2
"""Médecine""","""Séances simples""",2
"""Obstétrique""","""Accouchement normal mère""",1
"""Obstétrique""","""IMG & fausses couches""",1


In [86]:
# Enrichissement du pool candidat (lot E1) — codes DAS tabac/alcool/
# corpulence + contexte patient. Le pool sort de la préparation des données
# DÉJÀ enrichi : la génération fictomed (3.1a) n'enrichit plus rien.
if not ENRICHIR_SCENARIOS:
    print("ENRICHIR_SCENARIOS = False — pool candidat inchangé.")
elif "candidate_source" not in globals():
    print("Pas de pool candidat — exécuter le tirage stratifié d'abord.")
elif "enrichi" in candidate_source.columns:
    print("SKIP — pool déjà enrichi "
          f"({int(candidate_source['enrichi'].sum())}/{candidate_source.height} lignes).")
else:
    candidate_source = enrichir_candidats(
        candidate_source, seed=ENRICHISSEMENT_SEED,
        politique=POLITIQUE_ENRICHISSEMENT,
    )
    display(candidate_source.filter(pl.col("enrichi")).select(
        [c for c in ("DPEC", "agean", "sexe", "taille_cm", "poids_kg", "imc",
                     "tabac", "alcool", "codes_ajoutes")
         if c in candidate_source.columns]
    ))

Enrichissement : 11/14 ligne(s) enrichie(s), 3 exclue(s), 9 code(s) DAS ajouté(s).
Exclusions : préfixe O ×3
Codes ajoutés : E6603×4, E6604×2, F101×1, F1725×1, F17202×1


DPEC,agean,sexe,taille_cm,poids_kg,imc,tabac,alcool,codes_ajoutes
str,i32,str,i64,i64,f64,str,str,str
"""Séances simples""",76,"""1""",184,75,22.2,"""non-fumeur""","""usage nocif, environ 5 verres/…","""F101"""
"""Séances simples""",55,"""1""",184,82,24.2,"""non-fumeur""","""pas de mésusage""",""""""
"""HDJ médecine adultes""",62,"""1""",172,83,28.1,"""non-fumeur""","""pas de mésusage""","""E6603"""
"""HDJ médecine adultes""",25,"""2""",163,91,34.3,"""non-fumeur""","""pas de mésusage""","""E6604"""
"""Médecine adultes > 3 nuits""",61,"""1""",177,79,25.2,"""non-fumeur""","""pas de mésusage""","""E6603"""
…,…,…,…,…,…,…,…,…
"""Chirurgie adultes < 3 nuits""",29,"""1""",183,85,25.4,"""non-fumeur""","""pas de mésusage""","""E6603"""
"""Chirurgie adultes < 3 nuits""",36,"""1""",182,71,21.4,"""non-fumeur""","""pas de mésusage""",""""""
"""Chirurgie adultes > 3 nuits""",19,"""1""",173,71,23.7,"""non-fumeur""","""pas de mésusage""",""""""


## 3. Run du test

### Paramétrage du test courant

In [87]:
# --- Paramétrage des chemins : SEUL endroit où un test est désigné ---
TEST_NUM = "06"    # le test courant (étiquettes tabac/alcool = données, bloc H)
PREV_TEST = "05"   # test précédent de la chaîne (None pour un tout premier test)


### 3.1 Écriture des prompts sur le disque — SEEDING puis FIGEMENT (idempotent)

Ordre impératif : **seeding** (création des dossiers CRH depuis les scénarios)
puis **figement** (résolution du jeu par famille dans chaque dossier). Chaque
étape s'exécute si sa production manque, sinon elle skippe avec son état.

### Montage du jeu

**Le jeu du test s'édite LÀ : `tests/<TEST_NUM>/system/one_gen/` — un `.txt`
par famille clinique.** Il est monté par copie du jeu du **test précédent**
(la chaîne des tests est la chaîne des versions) ; pour un tout premier test,
amorçage depuis `work_modif_prompts/template_one_gen`. Un `regles_atih.yml`
présent dans le jeu est copié tel quel (hors périmètre). Aucun appel fictomed
dans cette section — file system uniquement.

**Contexte patient fourni (enrichissement)** — quand `ENRICHIR_SCENARIOS`
est actif, taille, poids, IMC et statuts tabac/alcool ne sont plus inventés
par le modèle : ils sont tirés en amont (lois Esteban/Obépi, codes F17/F10 et
E660x ajoutés en DAS) et fournis dans le user prompt — le prompt système du
jeu doit les RESTITUER fidèlement (bloc H du jeu de tests/05).


In [88]:

WORK_DIR = REPO_ROOT / "work_prompts"
TESTS_DIR = WORK_DIR / "tests"  # versionné et partagé (§9)
TD = TESTS_DIR / TEST_NUM

print("Test courant :", TD, "(existe)" if TD.is_dir() else "(à créer)")
print(
    "Jeu amont    :",
    TESTS_DIR / PREV_TEST / "system" / "one_gen"
    if PREV_TEST
    else REPO_ROOT / "work_modif_prompts" / "template_one_gen",
)



Test courant : /Users/remi/Documents/Stream/work_prompts/tests/05 (existe)
Jeu amont    : /Users/remi/Documents/Stream/work_prompts/tests/04/system/one_gen


In [89]:
# État du test courant
_sys_dir = TD / "system" / "one_gen"
_scen = scenario_dirs(TD) if TD.is_dir() else []
print("Test               :", TD, "— présent" if TD.is_dir() else "— à créer")
print("Jeu system/one_gen :", "présent" if _sys_dir.is_dir() else "absent")
print("Dossiers scénario  :", len(_scen), _scen[:5], "…" if len(_scen) > 5 else "")
if _scen:
    _fige = (TD / _scen[0] / "prompt_system_one_gen.txt").is_file()
    print("Figement (1er dossier, prompt_system_one_gen.txt) :",
          "présent" if _fige else "absent")

Test               : /Users/remi/Documents/Stream/work_prompts/tests/05 — présent
Jeu system/one_gen : présent
Dossiers scénario  : 0 [] 


In [90]:
# Montage du jeu par la chaîne — skip si déjà monté
_sys_dir = TD / "system" / "one_gen"
if _sys_dir.is_dir():
    print("SKIP — jeu déjà monté :", _sys_dir)
else:
    src = (
        TESTS_DIR / PREV_TEST / "system" / "one_gen"
        if PREV_TEST
        else REPO_ROOT / "work_modif_prompts" / "template_one_gen"
    )
    # (historique : tests/01 utilisait la position "first" — sans importance,
    #  le notebook vise les tests futurs)
    TD.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, _sys_dir)
    print("MONTÉ :", _sys_dir, "<-", src)

SKIP — jeu déjà monté : /Users/remi/Documents/Stream/work_prompts/tests/05/system/one_gen


In [91]:
# Génération fictomed — un scénario par ligne du pool candidat (QUOTAS).
# Ré-exécutable tant que le test n'est pas seedé : rien n'est écrit dans les
# dossiers scénario, seuls les fichiers de travail vont sous TD/.fictomed/.
_scen = scenario_dirs(TD) if TD.is_dir() else []
if _scen:
    print(f"SKIP — test déjà seedé : {len(_scen)} dossiers scénario.")
    _tirage = TD / ".fictomed" / "scenarios_fictomed_selected.parquet"
    if _tirage.is_file():
        # archive du tirage de CE test (scénarios générés, enrichissement
        # compris) — pas les profils sources
        _archive = pl.read_parquet(_tirage)
        display(_archive.select(
            [c for c in ("TPEC", "DPEC", *RECAP_TIRAGE_COLUMNS,
                         "taille_cm", "poids_kg", "imc", "tabac", "alcool",
                         "codes_ajoutes") if c in _archive.columns]
        ))
    else:
        print("(pas d'archive du tirage :", _tirage, ")")
elif "candidate_source" not in globals():
    print("Pas de pool candidat en mémoire — exécuter le tirage stratifié "
          "(paramétrage du run) d'abord.")
else:
    # fichiers de travail fictomed sous TD/.fictomed/ (caché => hors découverte)
    FICTOMED_DIR = TD / ".fictomed"
    (FICTOMED_DIR / "_backups").mkdir(parents=True, exist_ok=True)

    # garde défensive : le pool doit arriver ici DÉJÀ enrichi (cellule
    # d'enrichissement, fin de la préparation des données) — jamais
    # d'enrichissement silencieux dans la génération.
    if ENRICHIR_SCENARIOS and "enrichi" not in candidate_source.columns:
        raise RuntimeError(
            "Pool candidat non enrichi — exécuter la cellule d'enrichissement "
            "(section 2, après le tirage stratifié) avant la génération."
        )

    aphp_utils.write_fictomed_config(
        config_file=FICTOMED_DIR / "servers.yaml",
        project_root=REPO_ROOT,
        run_dir=FICTOMED_DIR,
    )

    _, _, selected_scenarios = aphp_utils.generate_and_select_fictomed_scenarios(
        candidate_source=candidate_source,
        config_file=FICTOMED_DIR / "servers.yaml",
        aphp_data_dir=REPO_ROOT / "data" / "aphp",
        paths={"backups": FICTOMED_DIR / "_backups"},
        scenario_filters=SCENARIO_FILTERS,
        target_n=candidate_source.height,
        random_selection=RANDOM_SELECTION,
        random_seed=RANDOM_SEED,
        run_dir=FICTOMED_DIR,
    )

    # contrôle avant écriture : les quotas ont-ils survécu à la génération ?
    _ctrl = [c for c in ("TPEC", "DPEC", *RECAP_TIRAGE_COLUMNS,
                         "taille_cm", "poids_kg", "imc", "tabac", "alcool",
                         "codes_ajoutes")
             if c in selected_scenarios.columns]
    display(selected_scenarios.select(_ctrl))
    if "DPEC" in selected_scenarios.columns:
        display(selected_scenarios.group_by("TPEC", "DPEC").len().sort(["TPEC", "DPEC"]))
    else:
        print("TPEC/DPEC non propagées par fictomed — réappliquer with_typologie "
              "sur selected_scenarios si le contrôle des quotas est requis.")
    print("Génération prête — contrôler le tableau ; l'écriture de la graine "
          "est la cellule suivante.")

Configuration fictomed : /Users/remi/Documents/Stream/work_prompts/tests/05/.fictomed/servers.yaml

pipelines:
  brest:
    data:
      input: /Users/remi/Documents/Stream/data/brest
      output: /Users/remi/Documents/Stream/work_prompts/tests/05/.fictomed/sorties/scenarios_brest
  aphp:
    data:
      input: /Users/remi/Documents/Stream/data/aphp
      output: /Users/remi/Documents/Stream/work_prompts/tests/05/.fictomed/sorties/scenarios
      referentials: /Users/remi/Documents/Stream/data/aphp/referentials

fictomed importé depuis : /Users/remi/Documents/Stream/work_modif_prompts/_dependencies/fictomed_prompt_work/fictomed/__init__.py
Dossier lu par fictomed : /Users/remi/Documents/Stream/data/aphp
Profiles actif          : /Users/remi/Documents/Stream/data/aphp/scenarios_bn_all_20260128.pq
Backup                  : /Users/remi/Documents/Stream/work_prompts/tests/05/.fictomed/_backups/scenarios_bn_all_20260128.pq.20260902_213619.backup
Vérification des données d'entrée pour le pip

Construction contexte:   0%|          | 0/3 [00:00<?, ?étape/s]

Construction scénarios: 100%|██████████| 3/3 [00:01<00:00,  2.51étape/s]


Scénarios sauvegardés dans /Users/remi/Documents/Stream/work_prompts/tests/05/.fictomed/sorties/scenarios/aphp_scenarios_14_20260902_213621.parquet
Profiles original restauré : /Users/remi/Documents/Stream/data/aphp/scenarios_bn_all_20260128.pq

Scénarios candidats générés : (14, 66)
source_row_id: 14 uniques / 14
source_scenario_id: 14 uniques / 14
generation_id: 14 uniques / 14

Scénarios retenus : (14, 66)
Écrit             : /Users/remi/Documents/Stream/work_prompts/tests/05/.fictomed/scenarios_fictomed_selected.parquet


TPEC,DPEC,generation_id,template_name,case_management_type,age,sexe,icd_primary_description,icd_primary_code,nb_associated,taille_cm,poids_kg,imc,tabac,alcool,codes_ajoutes
str,str,str,str,str,i64,i64,str,str,i64,i64,i64,f64,str,str,str
"""Chirurgie et interventionnel""","""Chirurgie adultes < 3 nuits""","""4d840277-643f-4527-a7f4-0d2c72…","""surgery_outpatient.txt""","""DP""",36,1,"""Autres affections précisées de…","""N328""",1,182,71,21.4,"""non-fumeur""","""pas de mésusage""",""""""
"""Médecine""","""Médecine adultes > 3 nuits""","""73576776-2107-4313-b2cb-1d78a0…","""medical_inpatient.txt""","""DP""",61,1,"""Autres accidents ischémiques c…","""G458""",2,177,79,25.2,"""non-fumeur""","""pas de mésusage""","""E6603"""
"""Médecine""","""Séances simples""","""809a4a2e-6559-475f-9f59-417575…","""medical_outpatient.txt""","""Z511""",55,1,"""Tumeur maligne du pancréas end…","""C254+8""",1,184,82,24.2,"""non-fumeur""","""pas de mésusage""",""""""
"""Chirurgie et interventionnel""","""Interventionnel adultes > 3 nu…","""82ce7dc0-f149-4050-8a8c-b7f16e…","""medical_inpatient.txt""","""DP""",60,1,"""Infarctus sous-endocardique (a…","""I2148""",3,174,79,26.1,"""ex-fumeur, sevré depuis 4 ans …","""pas de mésusage""","""F17202 E6603"""
"""Médecine""","""Médecine adultes > 3 nuits""","""33a6cca1-6450-490c-828d-5be7a9…","""medical_inpatient.txt""","""DP""",41,2,"""Autres pancréatites aiguës""","""K858""",4,159,77,30.5,"""tabagisme actif, 12 cigarettes…","""pas de mésusage""","""F1725 E6604"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Médecine""","""HDJ médecine adultes""","""8285b522-ae86-4624-bd35-5ad7a7…","""medical_outpatient.txt""","""DP""",25,2,"""Mise en place et ajustement d'…","""Z468""",2,163,91,34.3,"""non-fumeur""","""pas de mésusage""","""E6604"""
"""Chirurgie et interventionnel""","""Chirurgie adultes > 3 nuits""","""e8cb5271-f388-4f5b-b4b7-b3cb1f…","""surgery_inpatient.txt""","""DP""",46,2,"""Désunions d'une plaie opératoi…","""T8138""",5,163,53,19.9,"""non-fumeur""","""pas de mésusage""",""""""
"""Médecine""","""Séances simples""","""d8679b3a-8b0a-479e-8cf2-8ba873…","""medical_outpatient.txt""","""Z512""",76,1,"""Autres anémies par carence en …","""D508""",2,184,75,22.2,"""non-fumeur""","""usage nocif, environ 5 verres/…","""F101"""


TPEC,DPEC,len
str,str,u32
"""Chirurgie et interventionnel""","""Chirurgie adultes < 3 nuits""",2
"""Chirurgie et interventionnel""","""Chirurgie adultes > 3 nuits""",2
"""Chirurgie et interventionnel""","""Interventionnel adultes < 3 nu…",1
"""Chirurgie et interventionnel""","""Interventionnel adultes > 3 nu…",1
"""Médecine""","""HDJ médecine adultes""",2
"""Médecine""","""Médecine adultes > 3 nuits""",2
"""Médecine""","""Séances simples""",2
"""Obstétrique""","""Accouchement normal mère""",1
"""Obstétrique""","""IMG & fausses couches""",1


Génération prête — contrôler le tableau ; l'écriture de la graine est la cellule suivante.


In [92]:
# Écriture de la graine — après contrôle du tirage (cellule précédente)
_scen = scenario_dirs(TD) if TD.is_dir() else []
if _scen:
    print(f"SKIP — test déjà seedé : {len(_scen)} dossiers scénario.")
elif "selected_scenarios" not in globals():
    print("Pas de tirage en mémoire — exécuter d'abord la cellule de tirage.")
else:
    # prefix porté par le jeu : system/one_gen/prefix.txt prime sur fictomed
    _prefix_file = TD / "system" / "one_gen" / "prefix.txt"
    if _prefix_file.is_file():
        _prefix = _prefix_file.read_text(encoding="utf-8").rstrip("\n")
        selected_scenarios = selected_scenarios.with_columns(
            pl.lit(_prefix).alias("prefix")
        )
        print(f"Prefix REMPLACÉ par celui du jeu ({_prefix_file}) — "
              + (f"{len(_prefix)} caractère(s)." if _prefix
                 else "fichier vide : pas de prefix."))
    else:
        print(f"Prefix fictomed d'origine CONSERVÉ (pas de {_prefix_file.name} dans le jeu).")

    # user prompt : insertion du contexte patient (Stream uniquement — dans
    # fictomed, le template appellera bloc_contexte directement)
    _user_fn = user_fn_enrichi() if ENRICHIR_SCENARIOS else None
    print(
        "Scénarios créés :",
        seed_user_prompts(TD, selected_scenarios, user_fn=_user_fn,
                          seed_path=SOURCE_PROFILES_PATH),
    )

    if ENRICHIR_SCENARIOS:
        # trace des décisions dans test.json (champ notes, spec §2.2)
        _tj = TD / "test.json"
        _data = json.loads(_tj.read_text(encoding="utf-8"))
        _data["notes"] = (f"enrichissement: seed={ENRICHISSEMENT_SEED}, "
                          f"décisions={POLITIQUE_ENRICHISSEMENT}")
        _tj.write_text(json.dumps(_data, ensure_ascii=False, indent=2) + "\n",
                       encoding="utf-8")
        print("test.json annoté :", _data["notes"][:100], "…")

Prefix REMPLACÉ par celui du jeu (/Users/remi/Documents/Stream/work_prompts/tests/05/system/one_gen/prefix.txt) — 8 caractère(s).
Scénarios créés : ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013']
test.json annoté : enrichissement: seed=42, décisions=Politique(age_min=18, prefixes_exclusion=('O', 'Z94', 'T86'), p_t …


#### b) Figement — résolution du jeu par scénario

`SYSTEM_PROMPT_FILE` est le nom du prompt système figé dans chaque dossier.
Pour **itérer** après édition du jeu : changer `SYSTEM_PROMPT_FILE`
(ex. `prompt_system_one_gen_v2.txt`) et `OUT_FILE` en conséquence (section
3.2, ex. `crh_v2.txt`) — les variantes coexistent, rien n'est écrasé.


In [93]:
SYSTEM_PROMPT_FILE = "prompt_system_one_gen.txt"

# Figement — skip si le fichier figé est déjà présent dans tous les dossiers
_names = scenario_dirs(TD) if TD.is_dir() else []
_manquants = [n for n in _names if not (TD / n / SYSTEM_PROMPT_FILE).is_file()]
if not _names:
    print("Pas de dossiers scénario — seeder d'abord (3.1a).")
elif not _manquants:
    print(f"SKIP — {SYSTEM_PROMPT_FILE} déjà figé dans tous les dossiers.")
else:
    print("Scénarios servis :", copy_system_prompts(TD, "one_gen", dest=SYSTEM_PROMPT_FILE))

Scénarios servis : ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013']


#### c) Prompts partagés du vérificateur (optionnel)

Posés une fois par `write_prompts` — skip s'ils sont déjà présents partout.


In [94]:
_names = scenario_dirs(TD) if TD.is_dir() else []
for _fname, _text in (
    ("prompt_system_verif.txt", VERIF_SYSTEM),
    ("user_verification.txt", VERIF_USER),
):
    if not _names:
        print("Pas de dossiers scénario — seeder d'abord (3.1a).")
        break
    if all((TD / n / _fname).is_file() for n in _names):
        print(f"SKIP — {_fname} déjà présent dans tous les dossiers.")
    else:
        print(f"{_fname} :", write_prompts(TD, _fname, _text))

prompt_system_verif.txt : ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013']
user_verification.txt : ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013']


### 3.2 Génération — contrôle à sec puis run réel

Le contrôle à sec ne fait aucun appel API et aucune écriture — pas besoin de
clé. C'est aussi le **test de complétude** : `generate` échoue (`BenchError`)
au moindre fichier manquant, aucun dossier n'est sauté en silence. Le run réel
écrase `OUT_FILE` à chaque re-run (geste normal) ; chaque run réel ajoute son
entrée au journal `usage.json`.

Transport (cellule 1) : `sync` par défaut — un appel par scénario, quelques-uns
en parallèle, trois tentatives par requête, puis validation commune ; le
JSONL d'entrée et de sortie est archivé sous `batches/<stem de out>/` comme
pour le batch. Pour ne tester qu'un scénario avant le run complet :
`only=["0000"]` (run partiel, entrée `partial` au journal).


In [95]:
OUT_FILE = "crh_generation.txt"

dry = generate(
    TD,
    system=SYSTEM_PROMPT_FILE,
    user="user_generation.txt",
    out=OUT_FILE,
    client=None,  # inutile en dry-run
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
    dry_run=True,
)
show_first_prompt(dry)

Scénario : 0000 — famille : surgery_outpatient

============================ PROMPT SYSTÈME — 14470 caractère(s) ============================
Vous êtes chirurgien praticien. Votre tâche est de générer un compte rendu opératoire de chirurgie ambulatoire à partit d'un scénario clinique réalisé avec des codes de la classification internationale des maladies, l'acte réalisé selon la classification commune des actes médicaux (CCAM) et d'autres informations décrivant l'hospitalisation.


# Contexte : le codage CIM-10

La CIM-10 est une classification des maladies, elle peut se définir comme un ensemble organisé de rubriques dans lesquelles on range des entités morbides en fonction de certains critères établis. La CIM est utilisée pour transposer les diagnostics de maladies ou autres problèmes de santé en codes alphanumériques, ce qui facilite le stockage, la recherche et l'analyse des données. Elle est très utilisée en France, en particulier pour le codage des causes de décès et pour la décl

In [ ]:
client = mistral_client()  # échoue ici, clairement, si MISTRAL_API_KEY absente

cr = generate(
    TD,
    system=SYSTEM_PROMPT_FILE,
    user="user_generation.txt",
    out=OUT_FILE,
    client=client,
    transport=TRANSPORT,
    max_workers=MAX_WORKERS,
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
)
print(cr.usage)

### Lecture des CR

`show_crh("0007")` affiche le CR d'un scénario en **markdown rendu** dans le
notebook (champ `CR` déplié + dictionnaires de formulations) — pratique après
un run de masse (contrôle d'un échantillon) comme après un run unitaire
(`only=[...]`, itération par copie...). La cellule suivante écrit pour
**tout** le test un aperçu `.md` à côté de chaque `.txt` (ex.
`0007/crh_generation.md`, versionné avec le test) : ouvrir un fichier puis
« Markdown: Open Preview » (⇧⌘V).


In [60]:
# Lecture lisible des CR — rendu markdown inline (réutilise scripts/show_crh.py)
import importlib.util

_spec_show = importlib.util.spec_from_file_location(
    "show_crh", REPO_ROOT / "scripts" / "show_crh.py"
)
_show_mod = importlib.util.module_from_spec(_spec_show)
_spec_show.loader.exec_module(_show_mod)

from IPython.display import Markdown


def show_crh(scenario: str, out: str | None = None) -> None:
    """Affiche en markdown le CR d'un dossier scénario du test courant."""
    path = TD / scenario / (out or OUT_FILE)
    if not path.is_file():
        print("Pas encore généré :", path)
        return
    text = _show_mod.render(path)
    if text is None:
        print("CR illisible (JSON) :", path)
        return
    display(Markdown(text))


_names = scenario_dirs(TD)
if _names:
    show_crh(_names[0])
else:
    print("Pas encore de dossiers scénario dans", TD)

## Compte rendu opératoire

### Identification du patient
Nom : Rovel
Prénom : Raymond
Date de naissance : 17/03/2005

### Intervention
Appendicectomie par abord de la fosse iliaque droite

### Date de l'intervention
26/08/2024

### Diagnostic
Appendicite aiguë compliquée d'une perforation appendiculaire

### Technique utilisée
Sous anesthésie générale, le patient a été installé en décubitus dorsal. Une incision de McBurney a été réalisée en fosse iliaque droite. Après ouverture du péritoine, l’appendice a été identifié, congestif et perforé à sa base, avec présence d’un épanchement purulent localisé. L’appendice a été disséqué, ligaturé à sa base et réséqué. Une toilette péritonéale au sérum physiologique a été effectuée. Un drain de Redon a été mis en place dans le cul-de-sac de Douglas. Fermeture plan par plan sur drainage.

### Constatations per-opératoires
Appendice inflammatoire, perforé à sa base, avec présence d’un abcès localisé en fosse iliaque droite. Pas d’extension péritonéale généralisée. Pas de lésion associée visible sur le cæcum ou l’iléon terminal.

### Matériels utilisés
- Drain de Redon (calibre 10)
- Fils résorbables (Vicryl 3/0 et 2/0)
- Sérum physiologique pour lavage

### Chirurgien
Dr Olivier Gounet

-----

## Compte rendu d’hospitalisation

### Identification du patient
Nom : Rovel
Prénom : Raymond
Date de naissance : 17/03/2005
Service : Chirurgie Viscérale – Hôpital Nord - Assistance Publique – Hôpitaux de Marseille

### Motif d’hospitalisation
Admission aux urgences le 26/08/2024 pour douleurs abdominales fébriles évoluant depuis 48 heures, localisées en fosse iliaque droite, associées à des nausées et une hyperthermie à 38,5°C. Le tableau clinique évoquait une appendicite aiguë compliquée, motivant une prise en charge chirurgicale en urgence.

### Antécédents
- Médicaux : Aucun antécédent notable
- Chirurgicaux : Aucun
- Familiaux : Aucun antécédent familial particulier
- Allergies : Aucune allergie connue

### Mode de vie
Patient étudiant, non-fumeur, ne consommant pas d’alcool. Pas d’activité physique régulière. Vit chez ses parents.

### Histoire de la maladie
Raymond Rovel, 19 ans, a consulté aux urgences le 26/08/2024 pour des douleurs abdominales évoluant depuis deux jours, initialement péri-ombilicales puis localisées en fosse iliaque droite. Les douleurs étaient associées à des nausées et une fièvre à 38,5°C. À l’examen clinique, il présentait une défense en fosse iliaque droite, un signe de Blumberg positif et une hyperleucocytose à la numération formule sanguine. Une échographie abdominale a confirmé le diagnostic d’appendicite aiguë avec signes de complication locale. Une appendicectomie par voie ouverte a été réalisée en urgence le jour même.

### Examen clinique
À l’admission, le patient était conscient, orienté, avec un état général conservé. La température était à 38,5°C, la tension artérielle à 120/70 mmHg, la fréquence cardiaque à 90 battements par minute. L’abdomen était souple mais douloureux à la palpation en fosse iliaque droite, avec une défense localisée. Le reste de l’examen clinique était sans particularité. Le poids était stable à 68 kg.

### Examens complémentaires
- **Numération formule sanguine** : Hyperleucocytose à 15 000/mm³ avec polynucléose neutrophile (85%). Hémoglobine à 14 g/dL, plaquettes à 300 000/mm³.
- **CRP** : Élevée à 120 mg/L.
- **Échographie abdominale** : Appendice augmenté de volume, mesurant 12 mm de diamètre, avec paroi épaissie et présence d’un épanchement localisé en fosse iliaque droite. Pas de signe de péritonite généralisée.
- **Bilan hépatique et rénal** : Normal.

### Évolution pendant l'hospitalisation
Le patient a été opéré en urgence le 26/08/2024. Les suites opératoires ont été simples, avec une apyrexie obtenue dès le lendemain de l’intervention. Le drain de Redon a été retiré au deuxième jour postopératoire en l’absence de débit significatif. La reprise du transit a été observée au troisième jour. Les pansements ont été refaits quotidiennement, avec une cicatrisation satisfaisante. Une antibiothérapie par amoxicilline-acide clavulanique a été poursuivie pendant 5 jours. Le patient a été revu en consultation postopératoire à J3, avec un état clinique stable et une bonne évolution. Il a quitté le service le 29/08/2024 avec une ordonnance d’antalgiques et de soins infirmiers pour les pansements.

### Conclusion
Appendicectomie pour appendicite aiguë compliquée d’une perforation appendiculaire, réalisée sans complication peropératoire. Les suites postopératoires ont été simples, permettant un retour à domicile au troisième jour. Un suivi en consultation est prévu dans 15 jours pour contrôle de la cicatrisation.

Dr Olivier Gounet
Chirurgie Viscérale – Hôpital Nord - Assistance Publique – Hôpitaux de Marseille


---

## Formulations

```json
{
  "diagnostics": {
    "Appendicite aiguë, autre et non précisée (K35.8)": [
      "appendicite aiguë compliquée d'une perforation appendiculaire",
      "appendicite aiguë avec signes de complication locale",
      "appendicite aiguë compliquée"
    ]
  },
  "informations": {
    "Date entrée": [
      "26/08/2024"
    ],
    "Date de sortie": [
      "29/08/2024"
    ],
    "Service d'hospitalisation": [
      "Chirurgie Viscérale – Hôpital Nord - Assistance Publique – Hôpitaux de Marseille"
    ],
    "Intervention": [
      "Appendicectomie par abord de la fosse iliaque droite"
    ],
    "Date de l'intervention": [
      "26/08/2024"
    ],
    "Nom/Prénom du chirurgien": [
      "Dr Olivier Gounet"
    ],
    "Nom/Prénom du patient": [
      "Rovel Raymond"
    ],
    "Âge": [
      "19 ans"
    ],
    "Sexe": [
      "masculin"
    ],
    "État général": [
      "conscient, orienté, avec un état général conservé",
      "état clinique stable"
    ],
    "Poids": [
      "68 kg"
    ],
    "Statut gestationnel": [],
    "Gestité": [],
    "Comorbidités": [
      "Aucun antécédent notable"
    ],
    "Médicaments": [
      "amoxicilline-acide clavulanique",
      "antalgiques"
    ],
    "NFS": [
      "Hyperleucocytose à 15 000/mm³ avec polynucléose neutrophile (85%)",
      "Hémoglobine à 14 g/dL, plaquettes à 300 000/mm³"
    ],
    "Créatinine": [
      "Normal"
    ],
    "Bilan hépatique": [
      "Normal"
    ]
  }
}
```


In [61]:
!python {REPO_ROOT / "scripts" / "show_crh.py"} {TD} --md --out {OUT_FILE}

/Users/remi/Documents/Stream/work_prompts/tests/04/0000/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0001/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0002/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0003/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0004/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0005/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0006/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0007/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0008/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0009/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0010/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0011/crh_generation.md
/Users/remi/Documents/Stream/work_prompts/tests/04/0012/crh_generation.md
/Users/remi/Documents/Stream/work_prom

### 3.3 Vérificateur (optionnel)

Verdict nourri par les CR de la section 3.2 — depuis la mémoire (`cr.reports`)
ou le disque (`load_reports`). Reprise de session : kernel redémarré, rien en
mémoire, le disque fait foi — `strict=False` charge les scénarios déjà servis,
`only=` restreint le run à ceux-là (les autres dossiers restent intacts).


In [70]:
try:
    ctx = load_reports(TD, OUT_FILE)
except BenchError:
    _names = scenario_dirs(TD)
    ctx = pl.DataFrame(
        {
            "scenario": _names,
            "report": ["[CR généré — placeholder de contrôle à sec]"] * len(_names),
        }
    )
    print(f"Pas de {OUT_FILE} sur disque : contexte placeholder.")

dry_verif = generate(
    TD,
    system="prompt_system_verif.txt",
    user="user_verification.txt",
    out="verdict.txt",
    client=None,
    model=MODEL,
    max_tokens=MAX_TOKENS_SUMMARY,
    pricing=PRICING,
    context=ctx,
    context_header=VERIF_HEADER,
    context_footer=VERIF_FOOTER,
    dry_run=True,
)
show_first_prompt(dry_verif)

Scénario : 0000 — famille : delivery_inpatient_hospit

============================ PROMPT SYSTÈME — 260 caractère(s) ============================
Tu es un médecin DIM. On te fournit un compte rendu
hospitalier généré automatiquement. Vérifie sa cohérence clinique et sa
conformité aux règles de codage, puis rends un verdict structuré :
CONFORME ou NON CONFORME, suivi de la liste des anomalies constatées.

============================ PROMPT USER — 8668 caractère(s) ============================
Vérifie le compte rendu suivant et rends ton verdict.

### COMPTE RENDU À VÉRIFIER

Le compte rendu suivant respecte les élements suivants :
        - les diagnostics ont une formulation moins formelle que la définition du code
        - le plan du CRH est conforme aux recommandations. Les antécédents sont présentés sous forme de liste à puces, les autres sections sont rédigées.
        - Les dates sont cohérentes avec le scénario
        - Les informations obligatoires sont présentes en entête
 

In [71]:
client = mistral_client()

cr_disque = load_reports(TD, OUT_FILE, strict=False)
if cr_disque.height == 0:
    raise RuntimeError(f"Aucun {OUT_FILE} sur disque : lancer d'abord le run réel (3.2).")

verdicts = generate(
    TD,
    system="prompt_system_verif.txt",
    user="user_verification.txt",
    out="verdict.txt",
    client=client,
    transport=TRANSPORT,
    max_workers=MAX_WORKERS,
    model=MODEL,
    max_tokens=MAX_TOKENS_SUMMARY,
    pricing=PRICING,
    context=cr_disque,
    context_header=VERIF_HEADER,
    context_footer=VERIF_FOOTER,
    only=cr_disque["scenario"].to_list(),
)
print(verdicts.usage)

Batch Mistral 587d68ad-f76e-4465-9482-a520871e500f — 10 requête(s), modèle mistral-large-latest, JSONL : /Users/remi/Documents/Stream/work_prompts/tests/02/batches/verdict/batch_input_20260814_152305_303958.jsonl
Statut : SUCCESS — 10/10
Statut final : SUCCESS
Usage(n_requests=10, input_tokens=22575, output_tokens=13718, total_tokens=36293, input_cost_usd=0.00564375, output_cost_usd=0.010288499999999999, total_cost_usd=0.01593225)


## 4. Bilan — coûts et vérification mécanique

Journal **append-only** : chaque run réel ajoute une entrée, re-runs compris —
l'argent dépensé reste tracé même quand les sorties sont écrasées.
`committed_usd` = total engagé ; `current_usd` = coût de l'état courant.

À côté, le **journal CSV global** `work_prompts/usage_log.csv` (spec §7) :
une ligne par scénario et par run, tous tests confondus — pure observation,
`usage.json` reste la source de `summarize_costs`. La cellule de stats
rapides ci-dessous skippe s'il n'existe pas encore.

Puis les contrôles mécaniques hors modèle (stdlib uniquement) : structure des
dossiers, schéma des CRH, formulations. Code retour non nul si au moins un
échec.


In [ ]:
if TESTS_DIR.is_dir():
    for _td in sorted(TESTS_DIR.iterdir()):
        if _td.is_dir() and not _td.name.startswith("."):
            print(f"=== {_td.name} — {_td} ===")
            display(summarize_costs(_td))
else:
    print("Aucun test encore créé sous", TESTS_DIR)

In [ ]:
# Stats rapides sur le journal CSV global des appels (spec §7) — observation ;
# usage.json reste la source de summarize_costs. Skip propre si absent.
USAGE_LOG = REPO_ROOT / "work_prompts" / "usage_log.csv"
if USAGE_LOG.is_file():
    _log = pl.read_csv(
        USAGE_LOG,
        schema_overrides={"test": pl.String, "scenario": pl.String, "batch_id": pl.String},
    )
    _aggs = [
        pl.len().alias("n_lignes"),
        pl.col("input_tokens").sum(),
        pl.col("output_tokens").sum(),
        pl.col("cost_usd").sum().round(6),
    ]
    print(f"{USAGE_LOG.relative_to(REPO_ROOT)} — {_log.height} ligne(s)")
    print("Coût et tokens par (test, out) :")
    display(_log.group_by(["test", "out"]).agg(_aggs).sort(["test", "out"]))
    print(f"Test courant {TD.name} — par template :")
    display(
        _log.filter(pl.col("test") == TD.name)
        .group_by("template")
        .agg(_aggs)
        .sort("template")
    )
else:
    print("SKIP — pas encore de journal CSV :", USAGE_LOG)


In [ ]:
!python {REPO_ROOT / "scripts" / "check_crh.py"} {TD}

In [ ]:
_reancre = REPO_ROOT / "scripts" / "reancre_crh.py"
if _reancre.is_file():
    !python {_reancre} {TD}
else:
    print("SKIP — script absent (pas encore commité) :", _reancre)

## Annexe A — deux générations (non utilisé)

Le workflow 2-gen (résumé puis CR : deux `generate`, le `reports` du premier
nourrissant le `context` du second, spec §4) n'est **pas utilisé** — conservé
pour le jour venu. Positions : `first_gen` et `second_gen` (amorçage depuis
`work_modif_prompts/template_first_gen` / `template_second_gen`, puis chaîne
comme le workflow principal). Cellules volontairement non exécutables — copier
dans des cellules code pour les activer.

```python
# Graine (test dédié) puis montage des DEUX positions
seed_user_prompts(TD, selected_scenarios, seed_path=SOURCE_PROFILES_PATH)
shutil.copytree(REPO_ROOT / "work_modif_prompts" / "template_first_gen",
                TD / "system" / "first_gen")
shutil.copytree(REPO_ROOT / "work_modif_prompts" / "template_second_gen",
                TD / "system" / "second_gen")

# [édition des jeux dans TD/system/first_gen/ et TD/system/second_gen/]
print("Figé first_gen  :", copy_system_prompts(TD, "first_gen"))
print("Figé second_gen :", copy_system_prompts(TD, "second_gen"))

# Premier temps — contrôle à sec (complétude) puis run réel
dry2a = generate(TD, system="prompt_system_first_gen.txt",
                 user="user_generation.txt", out="crh_resume.txt",
                 client=None, model=MODEL, max_tokens=MAX_TOKENS_SUMMARY,
                 pricing=PRICING, prefix_text=FIRST_GEN_PREFIX, dry_run=True)
show_first_prompt(dry2a)

res1 = generate(TD, system="prompt_system_first_gen.txt",
                user="user_generation.txt", out="crh_resume.txt",
                client=mistral_client(), model=MODEL,
                max_tokens=MAX_TOKENS_SUMMARY, pricing=PRICING,
                prefix_text=FIRST_GEN_PREFIX)

# Second temps — le résumé nourrit le contexte (à sec : placeholder possible,
# même geste que la reprise du §8)
cr2 = generate(TD, system="prompt_system_second_gen.txt",
               user="user_generation.txt", out="crh_final.txt",
               client=mistral_client(), model=MODEL,
               max_tokens=MAX_TOKENS_CR, pricing=PRICING,
               prefix_file="prefix.txt",  # prefill d'origine de la graine
               context=res1.reports,      # ou load_reports(TD, "crh_resume.txt")
               context_header=SUMMARY_HEADER, context_footer=SUMMARY_FOOTER)
print(cr2.usage)
```


## Annexe B — `prompt_local.py` — logique de user prompt locale au test (§3.7)

Pour tester une **construction** de user prompt différente sans toucher au
package : un `prompt_local.py` à la racine du test (c'est un fichier : la
découverte l'ignore), chargé ici et passé en `user_fn=` à `seed_user_prompts`
d'un **nouveau** test — changer `TEST_NUM`/`PREV_TEST` en tête de notebook,
puis utiliser la graine ci-dessous **à la place** de celle de la section 3 ;
la suite (montage, figement, dry-run) est le workflow principal, inchangé.
Hiérarchie des leviers : (1) éditer les `.txt` du test ; (2) `prompt_local.py` ;
(3) monkeypatch fictomed (fragile, à noter dans `test.json["notes"]`) ;
(4) modifier le clone fictomed éditable.


In [ ]:
TD.mkdir(parents=True, exist_ok=True)

_prompt_local_path = TD / "prompt_local.py"
if not _prompt_local_path.exists():  # ne jamais écraser une version éditée
    _prompt_local_path.write_text(
        '''"""User prompt local au test (spec §3.7) — exemple.

Point de départ possible : inspect.getsource sur la fonction fictomed
correspondante, copiée puis modifiée.
"""


def build_user(row: dict) -> str:
    """User prompt fictomed + rappel explicite du DP et du GHM."""
    return (
        str(row["user_prompt"]).rstrip()
        + "\\n\\nRappel codage : DP "
        + str(row.get("icd_primary_code"))
        + " — GHM "
        + str(row.get("ghm2"))
        + "\\n"
    )
''',
        encoding="utf-8",
    )
    print("Écrit :", _prompt_local_path)

import importlib.util

_spec_local = importlib.util.spec_from_file_location(
    f"prompt_local_{TEST_NUM}", _prompt_local_path
)
prompt_local = importlib.util.module_from_spec(_spec_local)
_prev_dwb = sys.dont_write_bytecode
sys.dont_write_bytecode = True  # pas de __pycache__ dans le dossier de test
try:
    _spec_local.loader.exec_module(prompt_local)
finally:
    sys.dont_write_bytecode = _prev_dwb

print("Chargé :", prompt_local.build_user.__doc__)

In [ ]:
# Graine avec user_fn — À LA PLACE de la graine de la section 3.1, sur un test
# neuf. Refus normal (BenchError) si le test courant est déjà seedé.
if "selected_scenarios" not in globals():
    print("SKIP — pas de scénarios en mémoire (la chaîne fictomed n'a pas "
          "tourné : test déjà seedé).")
else:
    try:
        print(
            "Scénarios créés :",
            seed_user_prompts(
                TD,
                selected_scenarios,
                user_fn=prompt_local.build_user,
                seed_path=SOURCE_PROFILES_PATH,
            ),
        )
    except BenchError as exc:
        print("Graine refusée (test déjà seedé — normal en re-run) :", exc)

## Annexe C — itérer par copie d'un dossier scénario

Un dossier scénario est autonome : sa copie emporte **tous** ses prompts (et ses
sorties éventuelles). Nom libre ; le prochain `generate` sur le test l'inclut
dans la découverte — `only=` permet de ne relancer que la copie.


In [ ]:
_names = scenario_dirs(TD)
if not _names:
    print("Pas de dossiers scénario — seeder d'abord (3.1) ; copie skippée.")
else:
    _src = TD / _names[0]           # n'importe quel scénario servi
    _dst = TD / f"{_src.name}_bis"  # nom libre
    if _dst.exists():
        print("Copie déjà présente :", _dst)
    else:
        shutil.copytree(_src, _dst)
        print("Copié :", _src.name, "->", _dst.name)
    print("Découverte :", scenario_dirs(TD))

In [ ]:
# contrôle à sec — vérifie le prompt assemblé de la copie
if "_dst" not in globals() or not _dst.exists():
    print("Pas de copie de scénario — cellule précédente skippée.")
else:
    dry_copie = generate(
        TD,
        system="prompt_system_one_gen.txt",
        user="user_generation.txt",
        out="crh_generation.txt",
        client=None,  # inutile en dry-run
        model=MODEL,
        max_tokens=MAX_TOKENS_CR,
        pricing=PRICING,
        prefix_file="prefix.txt",
        only=[_dst.name],
        dry_run=True,
    )
    print(dry_copie.reports["system_prompt"][0][-1500:])

In [ ]:
# run réel, une seule requête — les autres dossiers restent intacts
cr_copie = generate(
    TD,
    system="prompt_system_one_gen.txt",
    user="user_generation.txt",
    out="crh_generation.txt",
    client=mistral_client(),  # la clé vient de l'environnement
    transport=TRANSPORT,
    max_workers=MAX_WORKERS,
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
    only=[_dst.name],
)
print(cr_copie.usage)

In [ ]:
if "_dst" in globals() and _dst.exists():
    show_crh(_dst.name)  # lecture du CR régénéré de la copie
else:
    print("Pas de copie de scénario — rien à lire.")

## Annexe D — tester l'ajout de DAS sur un scénario

`scenario_bis_avec_das("0002", ["I10", "E119"], suffix="t1")` copie le dossier
`0002` en `0002_t1` et ajoute les codes à la liste « Diagnostics associés » du
user prompt **avec leurs fiches descriptives** (référentiels fictomed :
fiche exacte, sinon fiche de catégorie ; codes déjà présents ou invalides
ignorés ; sorties purgées de la copie ; figement/template/prefix de la base
conservés). Le scénario de base reste intact — comparaison directe base/bis.
Puis `regen_crh("0002_t1")` régénère **ce seul CRH** (run réel : clé requise)
et écrit son `.md` à côté du `.txt`.


In [62]:
import re

In [63]:
# Test d'ajout de DAS : copie un dossier scénario en <base>_<suffix>, ajoute
# les codes à la liste « Diagnostics associés » du user prompt AVEC leurs
# fiches descriptives (référentiels fictomed), puis régénère ce seul CRH.
# Le scénario de base reste intact — la comparaison base/bis est directe.
from fictomed.sites.aphp.code_cards import CodeCardsRegistry, normalize_icd_code

_CARDS = CodeCardsRegistry.from_dirs(
    exact_dir=REPO_ROOT / "data/aphp/referentials/cards_library",
    category_dir=REPO_ROOT / "data/aphp/referentials/cards_library_categories",
)


def _libelle(card_text: str, code: str) -> str:
    m = re.search(r"^# \S+ — (.+)$", card_text, re.M)
    return m.group(1).strip() if m else f"Diagnostic associé ({code})"


def scenario_bis_avec_das(base: str, codes_das: list[str], *, suffix: str = "t1") -> str:
    """Crée le dossier `<base>_<suffix>` = scénario `base` + DAS ajoutés.

    - lignes ajoutées à « * Diagnostics associés : » (libellé + code) ;
    - fiches des nouveaux codes ajoutées à la fin (déjà présentes : skip) ;
    - sorties (crh, .md, verdict) purgées de la copie ;
    - figement, template, prefix : ceux de la base (copiés tels quels).
    """
    src, dst = TD / base, TD / f"{base}_{suffix}"
    if not src.is_dir():
        raise BenchError(f"Dossier scénario absent : {src}")
    if dst.exists():
        raise BenchError(f"{dst} existe déjà — choisir un autre suffix.")
    shutil.copytree(src, dst)
    for stale in dst.glob("crh_*"):
        stale.unlink()
    (dst / "verdict.txt").unlink(missing_ok=True)

    user_path = dst / "user_generation.txt"
    prompt = user_path.read_text(encoding="utf-8")

    lignes, fiches, sans_fiche = [], [], []
    for raw in codes_das:
        code = normalize_icd_code(raw)
        if not code:
            print(f"  {raw!r} : code invalide — ignoré")
            continue
        code_liste = code.replace(".", "")  # convention du scénario : sans point
        if f"({code_liste})" in prompt or f"({code})" in prompt:
            print(f"  {code_liste} : déjà dans le scénario — ignoré")
            continue
        card = _CARDS.find(code)
        if card is None:
            sans_fiche.append(code_liste)
            lignes.append(f"Diagnostic associé ajouté ({code_liste})")
            continue
        lignes.append(f"{_libelle(card.text, code)} ({code_liste})")
        if f'code="{card.card_code}"' not in prompt:
            fiches.append(card.text.strip())
    if not lignes:
        shutil.rmtree(dst)
        raise BenchError("Aucun code à ajouter — dossier bis non créé.")

    # 1. lignes DAS — sous « * Diagnostics associés : » (créée au besoin)
    m = re.search(r"^(\s*)\* Diagnostics associés :\s*$", prompt, re.M)
    if m:
        indent = m.group(1) + "   - "
        insertion = "".join(f"\n{indent}{l}" for l in lignes)
        prompt = prompt[: m.end()] + insertion + prompt[m.end():]
    else:
        m = re.search(r"^(\s*)\* Diagnostic principal :.*$", prompt, re.M)
        if not m:
            shutil.rmtree(dst)
            raise BenchError("Ancre « Diagnostic principal » introuvable dans le user prompt.")
        indent = m.group(1)
        bloc = f"\n{indent}* Diagnostics associés :" + "".join(
            f"\n{indent}   - {l}" for l in lignes
        )
        prompt = prompt[: m.end()] + bloc + prompt[m.end():]

    # 2. fiches des nouveaux codes — après la dernière fiche existante
    # (les fiches exactes ferment par </fiche_code>, les fiches de
    #  catégorie par </fiche_category>)
    if fiches:
        bloc = "\n\n" + "\n\n".join(fiches)
        fin, tag = max(
            (prompt.rfind(t), t) for t in ("</fiche_code>", "</fiche_category>")
        )
        if fin >= 0:
            fin += len(tag)
            prompt = prompt[:fin] + bloc + prompt[fin:]
        else:
            prompt = prompt.rstrip() + bloc + "\n"

    user_path.write_text(prompt, encoding="utf-8")
    print(f"Créé : {dst.name} — {len(lignes)} DAS ajouté(s), "
          f"{len(fiches)} fiche(s) insérée(s)"
          + (f", SANS fiche : {sans_fiche}" if sans_fiche else ""))
    return dst.name


def regen_crh(name: str, *, out: str | None = None) -> None:
    """Run réel du seul scénario `name`, puis .md à côté du .txt + lecture inline."""
    out = out or OUT_FILE
    cr = generate(
        TD,
        system=SYSTEM_PROMPT_FILE,
        user="user_generation.txt",
        out=out,
        client=mistral_client(),
        model=MODEL,
        max_tokens=MAX_TOKENS_CR,
        pricing=PRICING,
        prefix_file="prefix.txt",
        only=[name],
    )
    print(cr.usage)
    import subprocess

    subprocess.run(
        [sys.executable, str(REPO_ROOT / "scripts" / "show_crh.py"),
         str(TD / name), "--md", "--out", out],
        check=False,
    )
    show_crh(name, out=out)


E6603 Surpoids dû à un excès calorique, de l’adulte ou de l’enfant
E6604 Obésité due à un excès calorique de l’adulte avec indice de masse corporelle [IMC] égal ou supérieur à 30 kg/m² et inférieur à 35 kg m², ou obésité due à un excès calorique de l’enfant
E6605 Obésité due à un excès calorique de l’adulte avec indice de masse corporelle [IMC] égal ou supérieur à 35 kg/m² et inférieur à 40 kg/m²
E6606 Obésité due à un excès calorique de l’adulte avec indice de masse corporelle [IMC] égal ou supérieur à 40 kg/m² et inférieur à 50 kg/m²
E6607 Obésité due à un excès calorique de l’adulte avec indice de masse corporelle [IMC] égal ou supérieur à 50 kg/m²


F1720	Syndrome de dépendance au tabac, personne actuellement abstinente
F1724	Syndrome de dépendance au tabac, utilisation actuelle


F1020	Syndrome de dépendance à l'alcool, personne actuellement abstinente
F1024	Syndrome de dépendance à l'alcool, utilisation actuelle
F1026	Syndrome de dépendance à l'alcool, utilisation épisodique

In [64]:
# Exemple — à adapter puis décommenter :
nom = scenario_bis_avec_das("0008", ["E6600", "F1720","F1020"], suffix="t1")
regen_crh(nom)

Créé : 0009_t1 — 3 DAS ajouté(s), 3 fiche(s) insérée(s)
Batch Mistral 765b8bc3-83a5-4f21-a43f-6b0c081cd2d7 — 1 requête(s), modèle mistral-large-latest, JSONL : /Users/remi/Documents/Stream/work_prompts/tests/04/batches/crh_generation/batch_input_20260821_050617_766651.jsonl
Statut : SUCCESS — 1/1
Statut final : SUCCESS
Usage(n_requests=1, input_tokens=9248, output_tokens=1258, total_tokens=10506, input_cost_usd=0.002312, output_cost_usd=0.0009435, total_cost_usd=0.0032554999999999997)
/Users/remi/Documents/Stream/work_prompts/tests/04/0009_t1/crh_generation.md


Hôpital Pellegrin - CHU de Bordeaux
Service de Gynécologie

----------------------------------
Nom : Fonti
Prénom : Joulia
Date de naissance : 25/06/1982
----------------------------------

### Motif d'hospitalisation
Mme Joulia Fonti, âgée de 42 ans, a été admise ce jour en hospitalisation ambulatoire pour exploration et prise en charge d’un saignement génital anormal associé à des douleurs pelviennes modérées. Elle présente également des antécédents d’avortement spontané incomplet survenu il y a trois semaines, sans complication immédiate mais nécessitant un suivi rapproché.

### Antécédents
- Médicaux : Aucun antécédent médical notable en dehors d’un tabagisme actif et d’une consommation occasionnelle d’alcool.
- Chirurgicaux : Pas d’antécédent chirurgical.
- Familiaux : Pas d’antécédent familial particulier signalé.
- Allergies : Aucune allergie connue.

### Mode de vie
Mme Fonti est fumeuse depuis une vingtaine d’années, avec une consommation estimée à un paquet par jour. Elle rapporte également une consommation d’alcool occasionnelle, principalement le week-end. Elle travaille comme employée administrative et mène une vie sédentaire. Elle vit seule et n’a pas d’enfant. Son indice de masse corporelle est élevé, estimé à 32 kg/m², en lien avec une obésité modérée.

### Histoire de la maladie
La patiente décrit des saignements génitaux intermittents depuis environ deux mois, associés à des douleurs pelviennes modérées. Ces symptômes se sont accentués après un avortement spontané incomplet survenu il y a trois semaines, pour lequel elle n’a pas bénéficié d’une prise en charge chirurgicale. Elle consulte aujourd’hui pour évaluation diagnostique et thérapeutique. Aucun signe infectieux ni complication hémorragique sévère n’a été rapporté.

### Examen clinique
À l’admission, la patiente est apyrétique, avec une tension artérielle à 130/80 mmHg et une fréquence cardiaque à 78 battements par minute. Son poids est de 85 kg pour une taille de 1,62 m. L’examen gynécologique révèle un utérus légèrement augmenté de volume, sans masse palpable. Le col utérin est fermé, et il n’y a pas de saignement actif au moment de l’examen. L’examen abdominal est sans particularité, en dehors d’une sensibilité modérée à la palpation pelvienne.

### Examens complémentaires
Une vaginoscopie a été réalisée ce jour sous anesthésie locale. Elle a permis de visualiser une muqueuse utérine atrophique avec des zones de fibrose localisées. Aucun polype ni lésion suspecte n’a été observé. Les prélèvements histologiques ont été réalisés et seront analysés en anatomopathologie. Une échographie pelvienne réalisée en externe il y a une semaine montrait un utérus de taille normale mais avec un endomètre fin et irrégulier, sans épanchement pelvien.

### Conclusion
Mme Fonti présente une atrophie utérine avec fibrose, probablement secondaire à son avortement spontané récent. Une surveillance clinique et histologique est recommandée pour écarter toute autre pathologie sous-jacente. La patiente est autorisée à rentrer à domicile ce jour avec un rendez-vous de suivi dans un mois pour les résultats anatomopathologiques et une réévaluation clinique. Un sevrage tabagique et une prise en charge nutritionnelle sont également conseillés pour améliorer son état de santé général.

Bordeaux, le 14/05/2025

Dr Marie Rudloff


---

## Formulations

```json
{
  "diagnostics": {
    "Autres affections non inflammatoires précisées de l'utérus (N85.8)": [
      "une muqueuse utérine atrophique avec des zones de fibrose localisées",
      "une atrophie utérine avec fibrose"
    ],
    "Obésité due à un excès calorique (E66.00)": [
      "une obésité modérée"
    ],
    "Syndrome de dépendance au tabac (F17.20)": [
      "un tabagisme actif",
      "fumeuse depuis une vingtaine d’années"
    ],
    "Syndrome de dépendance à l'alcool (F10.20)": [
      "une consommation d’alcool occasionnelle"
    ],
    "Avortement spontané incomplet, sans complication (O03.4)": [
      "un avortement spontané incomplet survenu il y a trois semaines",
      "son avortement spontané récent"
    ]
  },
  "informations": {
    "Date entrée": [
      "14/05/2025"
    ],
    "Date de sortie": [
      "14/05/2025"
    ],
    "Service d'hospitalisation": [
      "Service de Gynécologie"
    ],
    "Nom/Prénom du médecin": [
      "Marie Rudloff"
    ],
    "Nom/Prénom du patient": [
      "Joulia Fonti"
    ],
    "Poids": [
      "85 kg"
    ],
    "Âge": [
      "42 ans"
    ],
    "Sexe": [
      "Féminin"
    ],
    "État général": [
      "La patiente est apyrétique, avec une tension artérielle à 130/80 mmHg et une fréquence cardiaque à 78 battements par minute."
    ],
    "NFS": [],
    "Créatinine": [],
    "Bilan hépatique": [],
    "Traitements": [
      "Un sevrage tabagique et une prise en charge nutritionnelle sont également conseillés"
    ]
  }
}
```
